Setting session and reading data from could(s3)

In [ ]:
from Scripts.config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

orders_df = spark.read.csv(
    s3_path("bronze", "orders", "olist_orders_dataset.csv"),
    header=True,
    inferSchema=True
)

orders_df.show(5)

understand the data before tranforming

In [ ]:

print("DATASET OVERVIEW")


print("Rows :", orders_df.count())
print("Columns :", len(orders_df.columns))


print("DUPLICATE ORDER IDS")


duplicate_orders = (
    orders_df.groupBy("order_id")
    .count()
    .filter("count > 1")
)

print("Duplicate Order IDs:", duplicate_orders.count())


print("NULL VALUES")
print("=" * 60)

from pyspark.sql.functions import *
orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()



print("ORDER STATUS")


orders_df.groupBy("order_status").count().show()

In [ ]:
from pyspark.sql.functions import col

orders_df.filter(
    col("order_approved_at").isNull()
).groupBy("order_status").count().show()

orders_df.filter(
    col("order_delivered_customer_date").isNull()
).groupBy("order_status").count().show()

orders_df.filter(
    col("order_delivered_carrier_date").isNull()
).groupBy("order_status").count().show()

In [ ]:
from pyspark.sql.functions import when, col

silver_orders = orders_df.withColumn(
    "data_quality_issue",
    when(
        (col("order_status") == "delivered") &
        (
            col("order_approved_at").isNull() |
            col("order_delivered_carrier_date").isNull() |
            col("order_delivered_customer_date").isNull()
        ),
        "MISSING_DELIVERY_INFORMATION"
    ).otherwise("OK")
)

silver_orders.columns


In [ ]:
silver_orders.filter(
    col("data_quality_issue") != "OK"
).show(truncate=False)